In [ ]:
# Importe le module sys pour connaître l'interpréteur Python actuellement utilisé.
import sys
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# Importe Path pour afficher le dossier courant du notebook.
from pathlib import Path

# Importe boto3, le SDK Python permettant de communiquer avec AWS.
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs


import pandas as pd

In [ ]:

s3 = fs.S3FileSystem(region="us-east-1")


# Indique le nom exact du bucket S3 contenant les données.
# Remplace cette valeur par le nom réel de ton bucket.
BUCKET_NAME = "mushroom-ml-mikael-2026-000"

# Équivalent AWS de PROJECT_ROOT.
# Dans notre cas, les fichiers Arrow sont dans le préfixe raw/.
PROJECT_ROOT = f"{BUCKET_NAME}/raw"

PARQUET_PATH = f"{PROJECT_ROOT}/parquet/mushrooms_merged.parquet"
PARQUET_OutPut = f"{PROJECT_ROOT}/parquet/mushrooms_CleanedNamesID.parquet"

table = pq.read_table(PARQUET_PATH,
    filesystem=s3)

print("Lignes :", table.num_rows)
print("Colonnes :", table.column_names)



In [ ]:
table.select(["label"]).slice(0, 10).to_pandas()

In [ ]:
labels = table["label"].to_pylist()

mushroom_names = []
edibilities = []

for label in labels:
    mushroom_name, edibility = label.rsplit("_", 1)

    mushroom_names.append(mushroom_name)
    edibilities.append(edibility)

print("Mushroom tables label orig , name, edibility")

In [ ]:
pd.DataFrame({
    "label_original": labels[:10],
    "mushroom_name": mushroom_names[:10],
    "edibility": edibilities[:10]
})

table = table.append_column("mushroom_name", pa.array(mushroom_names))
table = table.append_column("edibility", pa.array(edibilities))

In [ ]:
display(
    table.slice(0, 5).to_pandas()
)


entry_ids = [
    f"{index:06d}"
    for index in range(table.num_rows)
]

table = table.append_column(
    "entry_id",
    pa.array(entry_ids)
)

display(
    table.slice(0, 5).to_pandas()
)


In [ ]:
pq.write_table(
    table,
    PARQUET_OutPut,
    filesystem=s3,
    compression="zstd"
)